This file will generate a column that is populated by the cosine similarity of a child's description to the average adult description for a given category, both at half time (utterance_half_cossim_adults) and at full time (utterance_full_cossim_adults)

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
#Load data
utterances = pd.read_csv("../data/embeddings/utterance_half_and_full_embeddings.csv")

In [3]:
SEA_ANIMAL_CATEGORIES = ["crab", "octopus", "shark", "whale", "turtle", "seahorse"]

In [4]:
# Clean IDs/categories
utterances["participant_id"] = utterances["participant_id"].astype(str).str.strip()
utterances["target_category"] = utterances["target_category"].astype(str).str.strip().str.lower()

# Keep only sea animals
utterances = utterances[
    utterances["target_category"].isin(SEA_ANIMAL_CATEGORIES)
].copy()

In [5]:
def parse_embedding(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, np.ndarray):
        return x

    x = str(x).replace("\n", " ").strip("[]")
    embedding = np.fromstring(x, sep=" ")

    if embedding.size == 0:
        return np.nan

    return embedding

In [6]:
utterances["utterance_half_embedding"] = utterances["utterance_half_embedding"].apply(parse_embedding)
utterances["utterance_full_embedding"] = utterances["utterance_full_embedding"].apply(parse_embedding)


Helper functions

In [7]:
def average_embedding(embedding_series):
    valid_embeddings = [
        e for e in embedding_series
        if isinstance(e, np.ndarray) and e.size > 0
    ]

    if len(valid_embeddings) == 0:
        return None

    average = np.mean(np.vstack(valid_embeddings), axis=0)
    return average / np.linalg.norm(average)



In [8]:
def cosine_similarity_to_adult_average(row, embedding_col, adult_average_by_category):
    category = row["target_category"]
    individual_embedding = row[embedding_col]
    adult_average_embedding = adult_average_by_category.get(category)

    if not isinstance(individual_embedding, np.ndarray):
        return np.nan

    if adult_average_embedding is None:
        return np.nan

    return cosine_similarity(
        individual_embedding.reshape(1, -1),
        adult_average_embedding.reshape(1, -1)
    )[0, 0]

In [9]:
#Adults vs children
adult_mask = utterances["participant_id"].str.startswith("AD", na=False)
child_mask = utterances["participant_id"].str.startswith("BD", na=False)


In [10]:
#Compute adult prototypes

adult_half_embedding_by_category = (
    utterances.loc[adult_mask]
    .groupby("target_category")["utterance_half_embedding"]
    .apply(average_embedding)
    .to_dict()
)

adult_full_embedding_by_category = (
    utterances.loc[adult_mask]
    .groupby("target_category")["utterance_full_embedding"]
    .apply(average_embedding)
    .to_dict()
)

In [11]:
#Initialize output columns
utterances["utterance_half_cossim_adults"] = np.nan
utterances["utterance_full_cossim_adults"] = np.nan


In [12]:
#Compute cossim to adult average for children only
utterances.loc[child_mask, "utterance_half_cossim_adults"] = (
    utterances.loc[child_mask].apply(
        cosine_similarity_to_adult_average,
        axis=1,
        embedding_col="utterance_half_embedding",
        adult_average_by_category=adult_half_embedding_by_category
    )
)

utterances.loc[child_mask, "utterance_full_cossim_adults"] = (
    utterances.loc[child_mask].apply(
        cosine_similarity_to_adult_average,
        axis=1,
        embedding_col="utterance_full_embedding",
        adult_average_by_category=adult_full_embedding_by_category
    )
)

In [13]:
print(utterances["utterance_half_cossim_adults"].describe())
print(utterances["utterance_full_cossim_adults"].describe())

print("Adults:", adult_mask.sum())
print("Children:", child_mask.sum())

print(adult_half_embedding_by_category.keys())
print(adult_full_embedding_by_category.keys())

count    431.000000
mean       0.182057
std        0.223525
min       -0.315888
25%        0.021166
50%        0.163374
75%        0.352726
max        0.700126
Name: utterance_half_cossim_adults, dtype: float64
count    431.000000
mean       0.208818
std        0.205422
min       -0.306470
25%        0.061065
50%        0.210405
75%        0.351629
max        0.697445
Name: utterance_full_cossim_adults, dtype: float64
Adults: 120
Children: 527
dict_keys(['crab', 'octopus', 'seahorse', 'shark', 'turtle', 'whale'])
dict_keys(['crab', 'octopus', 'seahorse', 'shark', 'turtle', 'whale'])


In [14]:
print(
    utterances[
        ["participant_id", "target_category",
         "utterance_half_cossim_adults",
         "utterance_full_cossim_adults"]
    ].head(30)
)


   participant_id target_category  utterance_half_cossim_adults  \
0           AD001            crab                           NaN   
1           AD001           whale                           NaN   
2           AD001          turtle                           NaN   
3           AD001           shark                           NaN   
4           AD001        seahorse                           NaN   
5           AD001         octopus                           NaN   
6           AD002            crab                           NaN   
7           AD002         octopus                           NaN   
8           AD002           shark                           NaN   
9           AD002          turtle                           NaN   
10          AD002           whale                           NaN   
11          AD002        seahorse                           NaN   
12          AD003        seahorse                           NaN   
13          AD003         octopus                           Na

In [15]:
OUT_CSV = "../data/preprocessed/talk_cossim_adults.csv"

result = utterances.loc[child_mask, [
    "participant_id", "age_months", "target_category",
    "utterance_full_cossim_adults", "utterance_half_cossim_adults"
]].copy()

Path(OUT_CSV).parent.mkdir(parents=True, exist_ok=True)
result.to_csv(OUT_CSV, index=False)
print(f"Saved {result.shape[0]} rows → {OUT_CSV}")
print(result[["utterance_full_cossim_adults", "utterance_half_cossim_adults"]].describe().round(3))

Saved 527 rows → ../data/preprocessed/talk_cossim_adults.csv
       utterance_full_cossim_adults  utterance_half_cossim_adults
count                       431.000                       431.000
mean                          0.209                         0.182
std                           0.205                         0.224
min                          -0.306                        -0.316
25%                           0.061                         0.021
50%                           0.210                         0.163
75%                           0.352                         0.353
max                           0.697                         0.700
